In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.checkpoint.sqlite import SqliteSaver

memory = SqliteSaver.from_conn_string(":memory:")

In [3]:
from uuid import uuid4
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage

"""
In previous examples we've annotated the `messages` state key
with the default `operator.add` or `+` reducer, which always
appends new messages to the end of the existing messages array.

Now, to support replacing existing messages, we annotate the
`messages` key with a customer reducer function, which replaces
messages with the same `id`, and appends them otherwise.
"""
def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    # assign ids to messages that don't have them
    for message in right:
        if not message.id:
            message.id = str(uuid4())
    # merge the new messages with the existing messages
    merged = left.copy()
    for message in right:
        for i, existing in enumerate(merged):
            # replace any existing messages with the same id
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            # append any new messages to the end
            merged.append(message)
    return merged

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_messages]

In [4]:
tool = TavilySearchResults(max_results=2)

/var/folders/qn/bdtm30d94fq_77cy_0tjdzhc0000gn/T/ipykernel_17521/4289725543.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=2)


In [5]:
#Manual human approval
class Agent:
    def __init__(self, model, tools, system="", checkpointer=None):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"]
        )
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        print(state)
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [17]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-3.5-turbo")
with SqliteSaver.from_conn_string(":memory:") as memory:
    abot = Agent(model, [tool], system=prompt, checkpointer=memory)

    messages = [HumanMessage(content="Whats the weather in SF?")]
    thread = {"configurable": {"thread_id": "1"}}
    for event in abot.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v)

    abot.graph.get_state(thread)

    abot.graph.get_state(thread).next

    #Continue after interrupt
    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)

    abot.graph.get_state(thread)

    abot.graph.get_state(thread).next

    messages = [HumanMessage("Whats the weather in LA?")]
    thread = {"configurable": {"thread_id": "2"}}
    for event in abot.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v)

    while abot.graph.get_state(thread).next:
        print("\n", abot.graph.get_state(thread),"\n")
        _input = input("proceed?")
        if _input != "y":
            print("aborting")
            break
        for event in abot.graph.stream(None, thread):
            for v in event.values():
                print(v)

    #Modify State
    messages = [HumanMessage("Whats the weather in LA?")]
    thread = {"configurable": {"thread_id": "3"}}
    for event in abot.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v)

    abot.graph.get_state(thread)

    current_values = abot.graph.get_state(thread)

    current_values.values['messages'][-1]

    current_values.values['messages'][-1].tool_calls

    _id = current_values.values['messages'][-1].tool_calls[0]['id']
    current_values.values['messages'][-1].tool_calls = [
        {'name': 'tavily_search_results_json',
          'args': {'query': 'current weather in Louisiana'},
          'id': _id}
    ]

    abot.graph.update_state(thread, current_values.values)

    abot.graph.get_state(thread)

    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)

    #Time Travel
    states = []
    for state in abot.graph.get_state_history(thread):
        print(state)
        print('--')
        states.append(state)

    to_replay = states[-3]

    to_replay

    for event in abot.graph.stream(None, to_replay.config):
        for k, v in event.items():
            print(v)

    #Go back in time and edit
    to_replay

    _id = to_replay.values['messages'][-1].tool_calls[0]['id']
    to_replay.values['messages'][-1].tool_calls = [{'name': 'tavily_search_results_json',
          'args': {'query': 'current weather in LA, accuweather'},
                  'id': _id}]
    branch_state = abot.graph.update_state(to_replay.config, to_replay.values)

    for event in abot.graph.stream(None, branch_state):
        for k, v in event.items():
            if k != "__end__":
                print(v)

    #Add message to a state at a given time¶

    to_replay

    _id = to_replay.values['messages'][-1].tool_calls[0]['id']

    state_update = {"messages": [ToolMessage(
        tool_call_id=_id,
        name="tavily_search_results_json",
        content="54 degree celcius",
    )]}

    branch_and_add = abot.graph.update_state(
        to_replay.config, 
        state_update, 
        as_node="action")

    for event in abot.graph.stream(None, branch_and_add):
        for k, v in event.items():
            print(v)

{'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='6b7b2feb-4f05-4e2e-9e53-2d465b776681'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 152, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DWGY0AJNhKK6JAYwT7Lw3HJmxhi7q', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da491-3ee6-7f22-9cd7-80ad388732d8-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_8VThrbD4nTzuaSi6RNXRgo12', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inp

proceed? y


Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_5DM43m7NoFnUH0thWI0WmAbd', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'Weather Los Angeles in April 2026: Temperature & Climate\', \'url\': \'https://en.climate-data.org/north-america/united-states-of-america/california/los-angeles-714829/t/april-4/\', \'content\': \'| 18. April | Few clouds | 22 °C 71.6 °F | 16 °C 60.8 °F | 0 % | 14 km/h 9 mph | 0mm 0 in | 68% | [...] The first ten days of the month see average highs of 22.2°C | 72°F and lows around 9.2°C | 48.6°F. Between the 11th and 20th, highs average 23.7°C | 74.7°F, and lows are 9.9°C | 49.8°F. In the final third of the month, highs settle at 25.1°C | 77.2°F, with lows dipping to 11.2°C | 52.2°F.\\n\\n#### Which 10-day segment of April brings the highest average temperatures to Los Angeles?\\n\\nThe warmest period of 10 days is from 21-30, with highs averaging 17.6°C | 63.7°F.\\n\

In [18]:
#Extra Practice
#Build a small graph

from dotenv import load_dotenv

_ = load_dotenv()

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langgraph.checkpoint.sqlite import SqliteSaver

In [19]:
#Define a simple 2 node graph with the following state: -lnode: last node -scratch: a scratchpad location -count : a counter that is incremented each step

class AgentState(TypedDict):
    lnode: str
    scratch: str
    count: Annotated[int, operator.add]

def node1(state: AgentState):
    print(f"node1, count:{state['count']}")
    return {"lnode": "node_1",
            "count": 1,
           }
def node2(state: AgentState):
    print(f"node2, count:{state['count']}")
    return {"lnode": "node_2",
            "count": 1,
           }

In [20]:
#The graph goes N1->N2->N1... but breaks after count reaches 3.
def should_continue(state):
    return state["count"] < 3

In [21]:
builder = StateGraph(AgentState)
builder.add_node("Node1", node1)
builder.add_node("Node2", node2)

builder.add_edge("Node1", "Node2")
builder.add_conditional_edges("Node2", 
                              should_continue, 
                              {True: "Node1", False: END})
builder.set_entry_point("Node1")

In [27]:
with SqliteSaver.from_conn_string(":memory:") as memory:
    graph = builder.compile(checkpointer=memory)
    
    #Run it
    thread = {"configurable": {"thread_id": str(1)}}
    graph.invoke({"count":0, "scratch":"hi"},thread)

    #Look at current state
    graph.get_state(thread)

    #Look at state history
    for state in graph.get_state_history(thread):
        print(state, "\n")

    #Store just the config into an list. Note the sequence of counts on the right. get_state_history returns the most recent snapshots first.

    states = []
    for state in graph.get_state_history(thread):
        states.append(state.config)
        print(state.config, state.values['count'])

    #Grab an early state.
    states[-3]

    #This is the state after Node1 completed for the first time. Note next is Node2and count is 1.
    graph.get_state(states[-3])

    #Go Back in Time
    graph.invoke(None, states[-3])

    #Notice the new states are now in state history. Notice the counts on the far right.
    thread = {"configurable": {"thread_id": str(1)}}
    for state in graph.get_state_history(thread):
        print(state.config, state.values['count'])

    #You can see the details below. Lots of text, but try to find the node that start the new branch. Notice the parent config is not the previous entry in the stack, but is the entry from state[-3].
    thread = {"configurable": {"thread_id": str(1)}}
    for state in graph.get_state_history(thread):
        print(state,"\n")

    #Modify State
    #Let's start by starting a fresh thread and running to clean out history.
    thread2 = {"configurable": {"thread_id": str(2)}}
    graph.invoke({"count":0, "scratch":"hi"},thread2)

    from IPython.display import Image

    Image(graph.get_graph().draw_png())

    states2 = []
    for state in graph.get_state_history(thread2):
        states2.append(state.config)
        print(state.config, state.values['count']) 

    #start by grabbing a state
    save_state = graph.get_state(states2[-3])
    save_state

    #Now modify the values. One subtle item to note: Recall when agent state was defined, count used operator.add to indicate that values are added to the current value. Here, -3 will be added to the current count value rather than replace it.
    save_state.values["count"] = -3
    save_state.values["scratch"] = "hello"
    save_state

    #Now update the state. This creates a new entry at the top, or latest entry in memory. This will become the current state.
    graph.update_state(thread2,save_state.values)

    #Current state is at the top. You can match the thread_ts. Notice the parent_config, thread_ts of the new node - it is the previous node.

    for i, state in enumerate(graph.get_state_history(thread2)):
        if i >= 3:  #print latest 3
            break
        print(state, '\n')

    #Try again with as_node

    #When writing using update_state(), you want to define to the graph logic which node should be assumed as the writer. What this does is allow th graph logic to find the node on the graph. After writing the values, the next() value is computed by travesing the graph using the new state. In this case, the state we have was written by Node1. The graph can then compute the next state as being Node2. Note that in some graphs, this may involve going through conditional edges! Let's try this out.

    graph.update_state(thread2,save_state.values, as_node="Node1")

    for i, state in enumerate(graph.get_state_history(thread2)):
        if i >= 3:  #print latest 3
            break
        print(state, '\n')

    #invoke will run from the current state if not given a particular thread_ts. This is now the entry that was just added.
    graph.invoke(None,thread2)

    #Print out the state history, notice the scratch value change on the latest entries.
    for state in graph.get_state_history(thread2):
        print(state,"\n")

node1, count:0
node2, count:1
node1, count:2
node2, count:3
StateSnapshot(values={'lnode': 'node_2', 'scratch': 'hi', 'count': 4}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13bbff-1bf2-607c-8004-7c1ea26adf1a'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-04-19T07:18:20.582306+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13bbff-1bf0-64e8-8003-a1f38b404a49'}}, tasks=(), interrupts=()) 

StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hi', 'count': 3}, next=('Node2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13bbff-1bf0-64e8-8003-a1f38b404a49'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-04-19T07:18:20.581599+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13bbff-1bef-62aa-8002-61f1bdf2c886'}}, tasks=(PregelTask(id='e800a40a-ecf6